In [ ]:
!pip install transformers

In [ ]:
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data= pd.read_csv(r"/content/samsum-train.csv")
val_data= pd.read_csv(r"/content/samsum-validation.csv")

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(10)

,id,dialogue,summary
12048,13829299,Ron: Which one is your favorite harry potter?\...,Harry's favourite Harry Potter's books are the...
11028,13728222,Steve: Did you get tickets for the foam party ...,Max and Steve want to go to the foam party wit...
12469,13829334,Ricky: where's my dope\r\nJulian: fuck man don...,Ricky is asking Julian for his drug.
11110,13730455,Beth: Batman or Superman?\r\nConrad: neither -...,Conrad prefers Batman to Superman. Conrad send...
6812,13730594,David: it's foggy as hell this morning\r\nDavi...,It's foggy this morning. Rose got out of bed a...
3831,13716451,Laura: Are you guys packed?:D:D\r\nBruno: I am...,"Kim, Laura and Bruno are going for a trip tomo..."
11517,13680118,Caron: ahhh Ive found the Beautiful South on m...,Caron has found The Beautiful South on her Tid...
11310,13611886,Ann: I have got a plusnet mobile sim card that...,Ann and Rob are discussing ideas related to re...
531,13829328,"Bob: Hi, Madeleine.\r\nMadeleine: Hi, Bob, hav...",Bob has been traveling to Africa recently. He ...
4819,13731172,Meghan: You think Fong Ould like some coffee?\...,Meghan wonders if Fong Ould would like some co...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

##### Random Sampling

In [ ]:


train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

In [ ]:
val_data.shape

(500, 3)

#### Data Pre-processing

In [ ]:
import re

def clean_data(text):
  text= re. sub(r"\r\n", " ", text) # to remove extra line
  text= re. sub(r"\s+", " ", text) # to remove extra space
  text = re.sub(r"<.*?>", " ", text) # to remove html tags <p1><h1>
  text= text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)


In [ ]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
# Raw data ==> tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding ="max_length", max_length =512, truncation=True)
  targets = tokenizer(data["summary"], padding ="max_length", max_length =512, truncation=True)
  inputs["labels"]= targets["input_ids"]  # token ids ==> add to input as labels
  return inputs

In [ ]:
train_dataset = train_data.apply(tokenize , axis=1).tolist()
val_dataset = val_data.apply(tokenize , axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# Input ids

# 1's => End of Sequence
# attention mask
# labels - target ==> Summary token

In [ ]:
len(train_dataset[0]["input_ids"])

512

In [ ]:
type(train_dataset)
type(val_dataset)

list

#### working with our Model

In [ ]:
# NLP => Generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch
if torch.backends.mps.is_available():
  device = torch.device("mps")
elif torch.cuda.is_available():

  device = torch.device("cuda")
else:
  device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps = 500 # 0 => lr by default

)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# To train the Model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.850243,0.114187
2,0.120106,0.107293
3,0.112383,0.105169


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.850243,0.114187
2,0.120106,0.107293
3,0.112383,0.105169
4,0.109172,0.103926
5,0.106886,0.103476
6,0.106029,0.103286


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.7341365903218587, metrics={'train_runtime': 2256.2054, 'train_samples_per_second': 10.637, 'train_steps_per_second': 1.33, 'total_flos': 3248203235328000.0, 'train_loss': 0.7341365903218587, 'epoch': 6.0})

In [ ]:
# Model Load => Fine-tune ==> Save the model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

####  Test the core Logic for Summarization


In [ ]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) # clean

 # tokenize
 # Generate the Summary in the form of token ids
 # token ids convert to text summary
  inputs = tokenizer(
       dialogue,
       padding="max_length",
       truncation=True,
       max_length=512,
       return_tensors="pt"
)
 # Generate the Summary => token ids
  model.to(device)
  inputs = inputs.to(device)
  targets= model.generate(
       input_ids= inputs["input_ids"],
       attention_mask= inputs["attention_mask"].to(device),
        max_length=51,
        num_beams= 4,
        #repetition_penalty= 2.5,
        #length_penalty= 1.0,
        early_stopping= True
)

# token ids convert to summary => Decoding
  summary = tokenizer.decode(targets[0], skip_special_tokens=True)

  return summary







In [ ]:
test_dialogue ="""
Reporter: What is Artificial Intelligence?

Expert: Artificial Intelligence, or AI, is a field of computer science
that focuses on creating systems capable of performing tasks that normally
require human intelligence. These tasks include understanding language,
recognizing images, solving problems, making decisions, and learning from
experience.

Reporter: What is Machine Learning?

Expert: Machine Learning is a branch of Artificial Intelligence in which
computers learn patterns from data instead of being explicitly programmed
for every individual task. The system analyzes examples and uses what it
learns to make predictions or decisions.

Reporter: What is Natural Language Processing?

Expert: Natural Language Processing, or NLP, enables computers to understand,
process, analyze, and generate human language. NLP is used in chatbots,
translation, search engines, sentiment analysis, speech recognition, and
text summarization.

Reporter: What are Transformers?

Expert: Transformers are neural network architectures that use attention
mechanisms to process relationships between tokens in a sequence. They can
process many tokens efficiently and are widely used in modern NLP systems.

Reporter: Why are Transformers important in NLP?

Expert: Transformers have significantly improved NLP because their attention
mechanism allows models to understand relationships between different parts
of a text. They are used for translation, question answering, text
generation, and summarization.

Reporter: What is text summarization?

Expert: Text summarization automatically creates a shorter version of a
document while preserving its most important information. Extractive
summarization selects important sentences from the original text, while
abstractive summarization generates new sentences that represent the main
ideas.

Reporter: What are the benefits of AI-based summarization?

Expert: AI-based summarization can reduce large amounts of information into
a concise and understandable summary. It saves time and helps users quickly
understand important information from documents, meetings, reports, and
conversations.







"""

summary= summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  artificial intelligence, or ai, is a field of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. nlp is used in chatbots, translation, search engines, sentiment analysis,
